# D3 · L4 BOED Interactive on §9E.1 L4

**PHYRE W3 Track D · CPU 可跑 · ~15 min**

## 目标
在 §9E.1 L4 (17 题) 上跑 BOED 闭环：
1. prior over hypotheses (PRM 给的 softmax)
2. 候选 design $d$（SoC × 频段 × 温度网格）
3. EIG(d) = E_h E_{Z|h,d} log p(h|Z,d) − E_h log p(h)，用 nested MC 估计
4. 选 argmax_d EIG，调 D2 oracle (analytic surrogate) 拿 Z̃，更新 posterior
5. 评测：相比 random design，EIG 改善 ≥ 0.5 nats；posterior top-1 EM ≥ 0.55

## PASS
- ΔEIG (BOED − random) ≥ 0.5 nats（按题平均）
- posterior top-1 EM ≥ 0.55

In [ ]:
# ========== 1. setup ==========
import os, sys, json, time, math, warnings
from pathlib import Path
from itertools import product
import numpy as np
warnings.filterwarnings('ignore')

try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    PHYRE = Path('/content/drive/MyDrive/phyre')
except Exception:
    PHYRE = Path('phyre')

DATA  = PHYRE / 'data' / 'benchmark'
SRC   = PHYRE / 'src'
PVGAP = PHYRE / 'pvgap_experiment'
LOGS  = PHYRE / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
BENCH = DATA / 'echem_reason_benchmark.jsonl'
for p in (SRC, PVGAP, PHYRE): sys.path.insert(0, str(p))
np.random.seed(0)

with open(BENCH, encoding='utf-8') as f:
    BENCH_ALL = [json.loads(l) for l in f]
L4 = [e for e in BENCH_ALL if e.get('level') == 4]
print(f'§9E.1 L4 questions: {len(L4)} (target: 17)')

In [ ]:
# ========== 2. EIS surrogate + PRM (mirror C3) ==========
try:
    from pce_simulator_pybamm import eis_from_params
    print('loaded C2 analytic surrogate')
except ImportError:
    def eis_from_params(p, freqs_hz, soc=0.5):
        k = float(p.get('__scale_kappa_e__', 1.0))
        s = float(p.get('Negative electrode sei thickness [m]', 5e-9))
        r = float(p.get('Positive particle radius [m]', 5.22e-6))
        R0 = 0.1/max(k,1e-3); Rsei=1e8*s; Rct=1.0*(5.22e-6/max(r,1e-7))
        w = 2*np.pi*np.asarray(freqs_hz,float)
        return R0 + Rsei/(1+1j*w*Rsei*2e-3) + Rct/(1+1j*w*Rct*1e-1) + 0.1*(1-1j)/np.sqrt(np.maximum(w,1e-9))
    print('inline analytic surrogate')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
device = 'cuda' if torch.cuda.is_available() else 'cpu'

CKPT_DIR = PHYRE / 'ckpt'
CKPT_FOUND = None
for name in ('prm_v2.pt', 'prm_v1.pt', 'prm.pt', 'prm_best.pt'):
    if (CKPT_DIR / name).exists(): CKPT_FOUND = CKPT_DIR / name; break
if CKPT_FOUND is None and CKPT_DIR.exists():
    cands = sorted(CKPT_DIR.glob('*prm*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    CKPT_FOUND = cands[0] if cands else None
print(f'PRM ckpt: {CKPT_FOUND}')

tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')
prm = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-large', num_labels=1, ignore_mismatched_sizes=True)
if CKPT_FOUND is not None:
    try: state = torch.load(CKPT_FOUND, map_location=device, weights_only=False)
    except TypeError: state = torch.load(CKPT_FOUND, map_location=device)
    sd = state.get('model', state) if isinstance(state, dict) else state
    prm.load_state_dict(sd, strict=False)
prm.to(device).eval()

@torch.no_grad()
def prm_score(q, h):
    enc = tok(q + ' [SEP] ' + h, max_length=512, truncation=True, return_tensors='pt').to(device)
    return float(torch.sigmoid(prm(**enc).logits.squeeze()).item())

In [ ]:
# ========== 3. design space + hypothesis bucket ==========
import re
def _toks(s): return set(re.findall(r'[a-z]{3,}', (s or '').lower()))
def hyp_to_params(h):
    t = _toks(h); out = {}
    if t & {'sei','interfacial','passivation'}: out['Negative electrode sei thickness [m]'] = 1e-8
    if t & {'diffusion','warburg','particle','fracture'}: out['Positive particle radius [m]'] = 2.6e-6
    if (t & {'ohmic','electrolyte','resistance'}) and not (t & {'sei'}): out['__scale_kappa_e__'] = 0.5
    if t & {'plating','dendrite'}:
        out['Negative electrode sei thickness [m]'] = 1.5e-8; out['Positive particle radius [m]'] = 3.5e-6
    return out

def get_hyps(e):
    for k in ('hypotheses','candidates','options'):
        v = e.get(k)
        if isinstance(v,list) and v: return [str(h.get('text',h) if isinstance(h,dict) else h) for h in v]
    gt = e.get('ground_truth') or {}
    base = gt.get('hypothesis') or gt.get('mechanism') or 'sei growth'
    return [base, 'ohmic limited', 'particle fracture', 'lithium plating']

# design grid: SoC × freq band × T
SOCS  = [0.2, 0.5, 0.8]
BANDS = [(1e-3, 1e0), (1e-1, 1e2), (1e1, 1e4)]   # low/mid/high
TEMPS = [283.15, 298.15, 318.15]
DESIGNS = list(product(SOCS, BANDS, TEMPS))
print(f'design space: {len(DESIGNS)} (SoC×band×T = {len(SOCS)}×{len(BANDS)}×{len(TEMPS)})')

In [ ]:
# ========== 4. nested-MC EIG + posterior update ==========
SIGMA_LOGZ = 0.15   # observation noise on log|Z|
M_OUTER = 8         # h samples

def sample_freqs(band, n=15):
    return np.logspace(np.log10(band[0]), np.log10(band[1]), n)

def loglik(z_obs, z_mean):
    return -0.5 * np.sum(((np.log(np.abs(z_obs)+1e-9) - np.log(np.abs(z_mean)+1e-9))/SIGMA_LOGZ)**2)

def eig(question, H, p, design):
    soc, band, T = design
    fr = sample_freqs(band)
    Zh = [eis_from_params({**hyp_to_params(h), 'Ambient temperature [K]': T}, fr, soc=soc) for h in H]
    # outer: sample h ~ p, draw z; inner: log p(z|h')
    eig_acc = 0.0
    for _ in range(M_OUTER):
        i = int(np.random.choice(len(H), p=p))
        z = Zh[i] * np.exp(SIGMA_LOGZ * (np.random.randn(*Zh[i].shape) + 1j*np.random.randn(*Zh[i].shape))/np.sqrt(2))
        ll = np.array([loglik(z, Zh[j]) for j in range(len(H))])
        ll -= ll.max()
        post = p * np.exp(ll); post /= post.sum()
        eig_acc += np.sum(post * (np.log(post+1e-12) - np.log(p+1e-12)))
    return float(eig_acc / M_OUTER)

def update_posterior(question, H, p, design, gt_idx):
    soc, band, T = design
    fr = sample_freqs(band)
    Zh = [eis_from_params({**hyp_to_params(h), 'Ambient temperature [K]': T}, fr, soc=soc) for h in H]
    z_obs = Zh[gt_idx] * np.exp(SIGMA_LOGZ * np.random.randn(*Zh[gt_idx].shape))
    ll = np.array([loglik(z_obs, Z) for Z in Zh]); ll -= ll.max()
    post = p * np.exp(ll); post /= post.sum()
    return post

In [ ]:
# ========== 5. evaluate on §9E.1 L4 ==========
results, t0 = [], time.time()
for i, e in enumerate(L4):
    q = e['question_text']
    H = get_hyps(e)
    s = np.array([prm_score(q, h) for h in H]); p0 = np.exp(s - s.max()); p0 /= p0.sum()
    gt = ((e.get('ground_truth') or {}).get('hypothesis') or H[0]).strip().lower()
    gt_idx = int(np.argmax([float(h.strip().lower()==gt) for h in H])) if any(h.strip().lower()==gt for h in H) else 0
    # BOED: pick argmax EIG
    eigs = [eig(q, H, p0, d) for d in DESIGNS]
    d_star = DESIGNS[int(np.argmax(eigs))]; eig_star = float(np.max(eigs))
    # random baseline
    d_rand = DESIGNS[int(np.random.randint(len(DESIGNS)))]
    eig_rand = float(np.mean([eig(q, H, p0, d_rand) for _ in range(2)]))
    # update posterior with d_star
    post = update_posterior(q, H, p0, d_star, gt_idx)
    top1 = H[int(np.argmax(post))]; em = float(top1.strip().lower() == gt)
    results.append({'qid': e['qid'], 'eig_boed': eig_star, 'eig_rand': eig_rand,
                    'delta_eig': eig_star - eig_rand, 'post_em': em, 'top1': top1})
    if (i+1) % 5 == 0:
        print(f'  [{i+1}/{len(L4)}] ΔEIG={np.mean([r["delta_eig"] for r in results]):.3f}  '
              f'EM={np.mean([r["post_em"] for r in results]):.3f}  ({time.time()-t0:.0f}s)')

DEL = float(np.mean([r['delta_eig'] for r in results]))
EM  = float(np.mean([r['post_em'] for r in results]))
print(f'\n>>> §9E.1 L4 (n={len(results)})  ΔEIG={DEL:.3f} nats  EM={EM:.3f}  (targets ΔEIG≥0.5, EM≥0.55)')
print(f'    PASS: ΔEIG={DEL>=0.5}  EM={EM>=0.55}')

(LOGS / 'D3_l4_results.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in results), encoding='utf-8')
(LOGS / 'D3_summary.json').write_text(json.dumps({
    'n': len(results), 'delta_eig_nats': DEL, 'em': EM,
    'pass_eig': DEL >= 0.5, 'pass_em': EM >= 0.55,
    'design_space': len(DESIGNS), 'sigma_logz': SIGMA_LOGZ, 'm_outer': M_OUTER,
    'eis_backend': 'analytic_surrogate_via_C2',
}, indent=2), encoding='utf-8')
print('wrote logs/D3_l4_results.jsonl + D3_summary.json')

## Go / No-Go

**PASS:** ΔEIG ≥ 0.5 nats AND EM ≥ 0.55 on §9E.1 L4 (17 题)

**On ΔEIG fail:** design space 太粗 → 加 SoC step、加 freq band、加 amplitude (大信号 vs 小信号)。

**On EM fail:** SIGMA_LOGZ 设错 → 在 C4 真实 DFN 后用 residual 重新校准。

**Commit once green:**
```
git add nb/D3_l4_boed.ipynb logs/D3_summary.json
git commit -m 'D3: L4 BOED — ΔEIG=X.XX EM=X.XX on §9E.1 L4'
git tag w3-track-d3-boed
```